# Semana 01 · Ciclo de vida completo de un modelo con MLflow

En esta práctica llevarás un caso de clasificación desde los datos hasta una API local:

`datos → experimentos → comparación → gate → modelo ganador → Registry → API → observabilidad`

El objetivo no es conseguir el número más alto, sino construir una decisión **reproducible,
auditable y desplegable**. Usaremos Wine Quality como caso didáctico de clasificación
multiclase: predeciremos la calidad original y no debe interpretarse como garantía comercial.

**Entorno:** Databricks Free Edition, compute serverless y experimento asociado a este notebook.
No necesitas tarjeta de crédito, cluster propio, Docker, `uv`, `pyproject.toml` ni tokens.

## Mapa y criterios de terminación

Al acabar debes poder enseñar en MLflow:

1. un lote de al menos siete runs comparables, con datos, parámetros, métricas y artefactos;
2. una regla de selección escrita **antes** de mirar el test;
3. un único candidato evaluado sobre test y marcado como ganador;
4. versiones en Unity Catalog y un ciclo `Challenger → Champion → rollback`;
5. una API HTTP local que carga el `model.joblib` del run ganador y supera pruebas válidas e inválidas;
6. un run de despliegue con latencia, volumen de peticiones, errores y evidencia de la respuesta.

Tiempo recomendado: 45 min de construcción guiada + 120–180 min de práctica autónoma y debrief.

## 0. Dependencias

Ejecuta `%pip` al principio. Si Databricks reinicia Python, continúa desde la celda de imports.
Las dependencias se instalan en la sesión del notebook; no se crea ningún entorno o fichero de
proyecto en el repositorio.

In [0]:
%pip install -q --upgrade "mlflow[databricks]==3.14.0" "scikit-learn==1.9.0" "pandas==2.2.3" "matplotlib==3.11.1" "joblib==1.5.3" "xgboost==3.0.5"

## Cómo trabajar con la versión sin resolver

Ejecuta las celdas en orden y completa cada `TODO`. Los bloques se detienen con
`NotImplementedError` para que un fallo no genere evidencia engañosa. Consulta la guía del
alumno antes de mirar la solución. Mantén el test cerrado hasta haber elegido el ganador.

## 1. Configuración e identidad del lote

In [0]:
import json
import joblib
import re
import tempfile
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
import xgboost
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

STUDENT_ALIAS = "afp-student"
BATCH_ID = uuid.uuid4().hex[:8]
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15
assert abs(TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE - 1.0) < 1e-9
MIN_VALIDATION_F1_MACRO = 0.28
QUALITY_MIN = 3
QUALITY_MAX = 8
NUM_CLASSES = QUALITY_MAX - QUALITY_MIN + 1
# Esta ruta es relativa a semana1/modules/01-mlflow-databricks-foundations/notebooks/.
DATASET_PATH = "../../../data/raw/WineQT.csv"

# TODO 1: valida el alias con una expresión regular y configura el tracking de Databricks.
_alias_re = re.compile(r"^[a-z0-9][a-z0-9_-]{2,30}$")
assert _alias_re.match(STUDENT_ALIAS), (
    f"STUDENT_ALIAS '{STUDENT_ALIAS}' no cumple el formato requerido (minúsculas, dígitos, guiones)."
)

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
_EXPERIMENT_PATH = "/Users/alvarofigueplaza@gmail.com/icai-muiaap-operacion-de-modelos/semana1/modules/01-mlflow-databricks-foundations/notebooks/01_tracking_mlops"
mlflow.set_experiment(_EXPERIMENT_PATH)
experiment_id = mlflow.get_experiment_by_name(_EXPERIMENT_PATH).experiment_id
print(f"✅ Alias: {STUDENT_ALIAS}  |  Batch: {BATCH_ID}")
print(f"✅ Experiment: {_EXPERIMENT_PATH} (id={experiment_id})")
print(f"✅ MLflow {mlflow.__version__}  |  sklearn {sklearn.__version__}  |  xgboost {xgboost.__version__}")

✅ Alias: afp-student  |  Batch: ac62b4ad
✅ Experiment: /Users/alvarofigueplaza@gmail.com/icai-muiaap-operacion-de-modelos/semana1/modules/01-mlflow-databricks-foundations/notebooks/01_tracking_mlops (id=1189943837819291)
✅ MLflow 3.14.0  |  sklearn 1.9.0  |  xgboost 3.0.5


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


## 2. Contrato y controles de datos

Permite nulos porque el pipeline los imputará, pero detente si faltan columnas, el dataset está
vacío, falta el target o `Id` se cuela como predictor. Construye un `quality_report`
serializable como JSON y confirma que el target contiene al menos dos clases.

In [0]:
data = pd.read_csv(DATASET_PATH)
TARGET = "quality"
EXPECTED_FEATURES = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol",
]

# TODO 2: valida el contrato y crea X, y y quality_report.
assert len(data) > 0, "El dataset está vacío."
assert TARGET in data.columns, f"Falta la columna target '{TARGET}'."
missing_cols = [c for c in EXPECTED_FEATURES if c not in data.columns]
assert not missing_cols, f"Faltan columnas: {missing_cols}"
assert "Id" not in EXPECTED_FEATURES, "Id no debe ser predictor."

quality_values = sorted(data[TARGET].unique().tolist())
assert quality_values == list(range(QUALITY_MIN, QUALITY_MAX + 1)), (
    f"Se esperaban calidades {QUALITY_MIN}..{QUALITY_MAX}; recibidas {quality_values}."
)
assert data[TARGET].nunique() >= 2, "El target necesita al menos dos clases."

X = data[EXPECTED_FEATURES].copy()
y = data[TARGET].astype(int) - QUALITY_MIN  # quality_class: 0..5 para XGBoost

class_mapping = {i: q for i, q in enumerate(range(QUALITY_MIN, QUALITY_MAX + 1))}

quality_report = {
    "rows": int(len(data)),
    "duplicates": int(data.duplicated().sum()),
    "nulls_total": int(data[EXPECTED_FEATURES + [TARGET]].isnull().sum().sum()),
    "nulls_per_column": data[EXPECTED_FEATURES + [TARGET]].isnull().sum().to_dict(),
    "target_distribution_original": data[TARGET].value_counts().sort_index().to_dict(),
    "target_distribution_encoded": y.value_counts().sort_index().to_dict(),
    "class_mapping": class_mapping,
    "num_classes": int(y.nunique()),
    "feature_count": len(EXPECTED_FEATURES),
    "features": EXPECTED_FEATURES,
    "checks": {
        "dataset_not_empty": len(data) > 0,
        "target_present": TARGET in data.columns,
        "all_features_present": not missing_cols,
        "id_excluded": "Id" not in EXPECTED_FEATURES,
        "quality_range_valid": quality_values == list(range(QUALITY_MIN, QUALITY_MAX + 1)),
        "at_least_two_classes": data[TARGET].nunique() >= 2,
    },
}

print(json.dumps(quality_report, indent=2))
print(f"\nX shape: {X.shape}  |  y shape: {y.shape}")
print(f"Clases (quality_class): {sorted(y.unique())}")

{
  "rows": 1143,
  "duplicates": 0,
  "nulls_total": 0,
  "nulls_per_column": {
    "fixed acidity": 0,
    "volatile acidity": 0,
    "citric acid": 0,
    "residual sugar": 0,
    "chlorides": 0,
    "free sulfur dioxide": 0,
    "total sulfur dioxide": 0,
    "density": 0,
    "pH": 0,
    "sulphates": 0,
    "alcohol": 0,
    "quality": 0
  },
  "target_distribution_original": {
    "3": 6,
    "4": 33,
    "5": 483,
    "6": 462,
    "7": 143,
    "8": 16
  },
  "target_distribution_encoded": {
    "0": 6,
    "1": 33,
    "2": 483,
    "3": 462,
    "4": 143,
    "5": 16
  },
  "class_mapping": {
    "0": 3,
    "1": 4,
    "2": 5,
    "3": 6,
    "4": 7,
    "5": 8
  },
  "num_classes": 6,
  "feature_count": 11,
  "features": [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol"
  ],
  "checks": {
    "dataset_not_empt

## 3. Divide 70 % train, 15 % validación y 15 % test, siempre estratificado

In [0]:
# TODO 3: crea X_train, X_valid, X_test, y_train, y_valid e y_test.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full,
    test_size=VALIDATION_SIZE / (TRAIN_SIZE + VALIDATION_SIZE),
    random_state=RANDOM_STATE, stratify=y_train_full,
)

print(f"Train:      {X_train.shape[0]} filas ({X_train.shape[0]/len(X):.1%})")
print(f"Validation: {X_valid.shape[0]} filas ({X_valid.shape[0]/len(X):.1%})")
print(f"Test:       {X_test.shape[0]} filas ({X_test.shape[0]/len(X):.1%})")
print(f"Total:      {len(X)} filas")
assert X_train.shape[0] + X_valid.shape[0] + X_test.shape[0] == len(X)

Train:      799 filas (69.9%)
Validation: 172 filas (15.0%)
Test:       172 filas (15.0%)
Total:      1143 filas


## 4. Pipeline y funciones de evaluación

In [0]:
def build_pipeline(config: dict) -> Pipeline:
    cfg = dict(config)
    candidate_name = cfg.pop("candidate_name", "unnamed")
    model_type = cfg.pop("model_type", "extra_trees")

    if model_type == "extra_trees":
        estimator = ExtraTreesClassifier(**cfg)
    elif model_type == "xgboost":
        estimator = XGBClassifier(**cfg)
    else:
        raise ValueError(f"model_type desconocido: {model_type}")

    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", estimator),
    ])


def evaluate_classifier(model: Pipeline, features: pd.DataFrame, labels: pd.Series) -> dict:
    import time as _time
    t0 = _time.perf_counter()
    predictions = model.predict(features)
    elapsed = _time.perf_counter() - t0
    latency_per_row_ms = (elapsed / len(features)) * 1000 if len(features) > 0 else 0.0

    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision_macro": float(precision_score(labels, predictions, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(labels, predictions, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
        "latency_per_row_ms": float(latency_per_row_ms),
        "predictions": predictions,
    }

## 5. Diseña al menos siete candidatos comparables

In [0]:
# TODO 5: crea al menos siete configuraciones comparables.
CANDIDATES = [
    {
        "candidate_name": "extra_trees_300_leaf1",
        "model_type": "extra_trees",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": 1.0,
        "class_weight": None,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "extra_trees_300_leaf2",
        "model_type": "extra_trees",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": 1.0,
        "class_weight": None,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "extra_trees_300_leaf3",
        "model_type": "extra_trees",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 3,
        "max_features": 1.0,
        "class_weight": None,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "extra_trees_600_leaf1",
        "model_type": "extra_trees",
        "n_estimators": 600,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": 1.0,
        "class_weight": None,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "extra_trees_600_leaf2",
        "model_type": "extra_trees",
        "n_estimators": 600,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": 1.0,
        "class_weight": None,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "xgboost_250_depth4",
        "model_type": "xgboost",
        "n_estimators": 250,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 1,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "objective": "multi:softprob",
        "num_class": NUM_CLASSES,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": 0,
    },
    {
        "candidate_name": "xgboost_300_depth5",
        "model_type": "xgboost",
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.03,
        "min_child_weight": 2,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 1.5,
        "objective": "multi:softprob",
        "num_class": NUM_CLASSES,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": 0,
    },
]
assert len(CANDIDATES) >= 7, "Necesitas al menos siete candidatos, incluyendo XGBoost."
assert any(candidate.get("model_type") == "xgboost" for candidate in CANDIDATES)
print(f"{len(CANDIDATES)} candidatos definidos.")
for c in CANDIDATES:
    print(f"  - {c['candidate_name']} ({c['model_type']})")

7 candidatos definidos.
  - extra_trees_300_leaf1 (extra_trees)
  - extra_trees_300_leaf2 (extra_trees)
  - extra_trees_300_leaf3 (extra_trees)
  - extra_trees_600_leaf1 (extra_trees)
  - extra_trees_600_leaf2 (extra_trees)
  - xgboost_250_depth4 (xgboost)
  - xgboost_300_depth5 (xgboost)


## 6. Tracking profundo: un run por candidato

Por cada candidato registra tags, parámetros, versiones, inputs de train/validación, tarjeta de
datos, calidad, riesgos, métricas, reporte por clase, matriz de confusión, modelo MLflow con
firma/input example, `deployment/model.joblib` y contrato de inferencia.

In [0]:
dataset_card = {
    "source": "WineQT.csv (UCI Wine Quality, variante roja, con Id)",
    "purpose": "Clasificación multiclase didáctica de calidad de vino (3..8) para practicar el ciclo de vida con MLflow.",
    "features": EXPECTED_FEATURES,
    "target": "quality_class",
    "target_definition": f"quality original 3..8; quality_class = quality - {QUALITY_MIN}",
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15, "random_state": RANDOM_STATE},
    "known_limitations": [
        "Clases extremas (3 y 8) tienen muy pocos ejemplos (6 y 16 respectivamente).",
        "No se trata de un dataset representativo de producción; es un caso didáctico.",
        "Las features fisicoquímicas no capturan factores humanos como método de elaboración o añejamiento.",
    ],
}
risk_register = {
    "version": "s01-lifecycle",
    "risks": [
        {"id": "R1", "risk": "Desbalance severo en clases extremas (quality 3: 6 muestras)", "impact": "high", "owner": STUDENT_ALIAS, "mitigation": "Usar F1 macro como métrica principal; no rebajar el gate a posteriori."},
        {"id": "R2", "risk": "Fuga de información: usar test antes de seleccionar el ganador", "impact": "high", "owner": STUDENT_ALIAS, "mitigation": "Selección exclusivamente sobre validación; test se abre una sola vez."},
        {"id": "R3", "risk": "No reproducibilidad por ausencia de semillas o versiones", "impact": "high", "owner": STUDENT_ALIAS, "mitigation": "Semillas explícitas (RANDOM_STATE=42) y versiones fijadas en %pip."},
        {"id": "R4", "risk": "Contrato de inferencia roto en serving (columnas o tipos distintos)", "impact": "high", "owner": STUDENT_ALIAS, "mitigation": "Firma MLflow + inference_contract.json validado en cada run."},
    ],
}
assert len(dataset_card["known_limitations"]) >= 3
assert len(risk_register["risks"]) >= 4

### 6a. Inputs de datos

Crea dos objetos `mlflow.data` con nombres distintos. Comprueba sus nombres antes de lanzar runs.

In [0]:
# TODO 6a: crea train_dataset y validation_dataset con mlflow.data.from_pandas.
train_df = X_train.copy()
train_df["quality_class"] = y_train.values

validation_df = X_valid.copy()
validation_df["quality_class"] = y_valid.values

train_dataset = mlflow.data.from_pandas(
    train_df, source=DATASET_PATH, targets="quality_class", name="wine_train",
)
validation_dataset = mlflow.data.from_pandas(
    validation_df, source=DATASET_PATH, targets="quality_class", name="wine_validation",
)

print(f"Train dataset:      name={train_dataset.name}, source={train_dataset.source}")
print(f"Validation dataset: name={validation_dataset.name}, source={validation_dataset.source}")
assert train_dataset.name != validation_dataset.name

Train dataset:      name=wine_train, source=<mlflow.data.artifact_dataset_sources.LocalArtifactDatasetSource object at 0xffcbad215bb0>
Validation dataset: name=wine_validation, source=<mlflow.data.artifact_dataset_sources.LocalArtifactDatasetSource object at 0xffcbad4238c0>


/local_disk0/.ephemeral_nfs/envs/pythonEnv-e33d8c8c-2d31-4d7f-b8ea-702894a98e4f/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-e33d8c8c-2d31-4d7f-b8ea-702894a98e4f/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


### 6b. Artefactos del candidato

Implementa una función pequeña que registre firma/input example, `model_info.model_uri`, Joblib,
tamaño, matriz de confusión y contrato. Podrás probarla dentro de un único run antes del bucle.

In [0]:
def log_candidate_artifacts(
    model: Pipeline, predictions: np.ndarray, candidate_index: int
) -> str:
    # Log model with sklearn flavor (Pipeline contains imputer + estimator)
    input_example = X_train.head(3)
    signature = infer_signature(input_example, model.predict(input_example))

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        signature=signature,
        input_example=input_example,
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.tree._tree.Tree",
            "xgboost.core.Booster",
            "xgboost.sklearn.XGBClassifier",
        ],
    )
    model_uri = model_info.model_uri
    mlflow.set_tag("candidate.model_uri", model_uri)

    # Serialize with joblib and log as deployment artifact
    with tempfile.TemporaryDirectory() as tmpdir:
        joblib_path = Path(tmpdir) / "model.joblib"
        joblib.dump(model, joblib_path)
        joblib_size = joblib_path.stat().st_size
        mlflow.log_artifact(str(joblib_path), artifact_path="deployment")

    # Confusion matrix
    labels_sorted = sorted(np.unique(np.concatenate([y_valid.values, predictions])))
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_valid.values, predictions, labels=labels_sorted, ax=ax,
        display_labels=[class_mapping[l] for l in labels_sorted],
    )
    ax.set_title(f"Confusion Matrix - Candidate #{candidate_index}")
    plt.tight_layout()
    mlflow.log_figure(fig, f"artifacts/confusion_matrix_candidate_{candidate_index}.png")
    plt.close(fig)

    # Inference contract
    inference_contract = {
        "input_features": EXPECTED_FEATURES,
        "input_types": {col: str(X_train[col].dtype) for col in EXPECTED_FEATURES},
        "output": "quality_class (int 0..5)",
        "output_mapping": class_mapping,
        "model_uri": model_uri,
        "joblib_path": "deployment/model.joblib",
        "joblib_size_bytes": int(joblib_size),
    }
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
        json.dump(inference_contract, f, indent=2)
        f.flush()
        mlflow.log_artifact(f.name, artifact_path="deployment")
        mlflow.log_artifact(f.name, artifact_path="")

    mlflow.log_metric("joblib_size_bytes", float(joblib_size))
    return model_uri

### 6c. Runs comparables

Ahora el bucle se ocupa sólo de identidad, parámetros, inputs, gobierno, entrenamiento, métricas
y de llamar al helper de artefactos. Conserva `experiment_id` y cada `run_id`.

In [0]:
experiment_id = experiment_id  # ya definido en la celda de configuración
candidate_run_ids = []

for idx, config in enumerate(CANDIDATES):
    run_name = f"{STUDENT_ALIAS}_{BATCH_ID}_c{idx:02d}_{config['candidate_name']}"
    with mlflow.start_run(experiment_id=experiment_id, run_name=run_name) as run:
        run_id = run.info.run_id
        candidate_run_ids.append(run_id)

        # Tags de identidad
        mlflow.set_tags({
            "student_alias": STUDENT_ALIAS,
            "batch_id": BATCH_ID,
            "candidate_name": config["candidate_name"],
            "model_type": config["model_type"],
            "workflow": "s01-lifecycle",
        })

        # Log JSON artifacts (governance)
        with tempfile.NamedTemporaryFile(mode="w", suffix="_dataset_card.json", delete=False) as f:
            json.dump(dataset_card, f, indent=2)
            f.flush()
            mlflow.log_artifact(f.name, artifact_path="governance")
        with tempfile.NamedTemporaryFile(mode="w", suffix="_risk_register.json", delete=False) as f:
            json.dump(risk_register, f, indent=2)
            f.flush()
            mlflow.log_artifact(f.name, artifact_path="governance")
        with tempfile.NamedTemporaryFile(mode="w", suffix="_quality_report.json", delete=False) as f:
            json.dump(quality_report, f, indent=2)
            f.flush()
            mlflow.log_artifact(f.name, artifact_path="governance")

        # Log versiones
        mlflow.log_param("mlflow_version", mlflow.__version__)
        mlflow.log_param("sklearn_version", sklearn.__version__)
        mlflow.log_param("xgboost_version", xgboost.__version__)

        # Log parámetros del candidato (excluyendo candidate_name y model_type)
        for key, val in config.items():
            if key not in ("candidate_name", "model_type"):
                mlflow.log_param(key, str(val) if val is not None else "None")
        mlflow.log_param("num_classes", NUM_CLASSES)
        mlflow.log_param("train_size", TRAIN_SIZE)
        mlflow.log_param("validation_size", VALIDATION_SIZE)

        # Log inputs de datos
        mlflow.log_input(train_dataset, context="training")
        mlflow.log_input(validation_dataset, context="validation")

        # Entrenar
        pipeline = build_pipeline(config)
        pipeline.fit(X_train, y_train)

        # Evaluar sobre validación
        val_metrics = evaluate_classifier(pipeline, X_valid, y_valid)
        predictions = val_metrics.pop("predictions")

        # Log métricas de validación
        mlflow.log_metrics({f"validation.{k}": v for k, v in val_metrics.items()})

        # Classification report
        report = classification_report(
            y_valid, predictions, output_dict=True, zero_division=0,
        )
        with tempfile.NamedTemporaryFile(mode="w", suffix="_classification_report.json", delete=False) as f:
            json.dump(report, f, indent=2)
            f.flush()
            mlflow.log_artifact(f.name, artifact_path="metrics")

        # Log artefactos del candidato
        model_uri = log_candidate_artifacts(pipeline, predictions, idx)

        # Fila de resumen
        print(
            f"  [{idx:02d}] {config['candidate_name']:30s} "
            f"f1_macro={val_metrics['f1_macro']:.4f}  "
            f"acc={val_metrics['accuracy']:.4f}  "
            f"lat={val_metrics['latency_per_row_ms']:.3f}ms  "
            f"run={run_id[:8]}"
        )

assert len(candidate_run_ids) == len(CANDIDATES)
print(f"\n✅ {len(candidate_run_ids)} runs registrados en experiment {experiment_id}")

2026/09/29 17:58:03 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-e33d8c8c-2d31-4d7f-b8ea-702894a98e4f/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ 

---------------------------------------------------------------------------
UntrustedTypesFoundException              Traceback (most recent call last)
File /local_disk0/.ephemeral_nfs/envs/pythonEnv-e33d8c8c-2d31-4d7f-b8ea-702894a98e4f/lib/python3.12/site-packages/mlflow/sklearn/__init__.py:683, in _save_model(sk_model, output_path, serialization_format, skops_trusted_types)
    682     skops.io.dump(sk_model, output_path)
--> 683     skops.io.load(output_path, trusted=skops_trusted_types)
    684 except UntrustedTypesFoundException as e:

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-e33d8c8c-2d31-4d7f-b8ea-702894a98e4f/lib/python3.12/site-packages/skops/io/_persist.py:152, in load(file, trusted)
    151 tree = get_tree(schema, load_context, trusted=trusted)
--> 152 audit_tree(tree, trusted=trusted)
    153 instance = tree.construct()

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-e33d8c8c-2d31-4d7f-b8ea-702894a98e4f/lib/python3.12/site-packages/skops/io/_audit.py:63, in audit_tr

## 7. Busca y compara exclusivamente los runs de tu alias y `BATCH_ID`

In [0]:
# TODO 7: usa mlflow.search_runs con tres tags en filter_string y muestra parámetros,
# métricas de validación, latencia y run_id en un DataFrame `comparison`.
filter_string = (
    f"tags.student_alias = '{STUDENT_ALIAS}' "
    f"AND tags.batch_id = '{BATCH_ID}' "
    f"AND tags.workflow = 's01-lifecycle'"
)

all_runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string=filter_string,
)

comparison = all_runs[[
    "tags.candidate_name",
    "tags.model_type",
    "params.n_estimators",
    "params.min_samples_leaf",
    "params.max_depth",
    "params.learning_rate",
    "metrics.validation.f1_macro",
    "metrics.validation.f1_weighted",
    "metrics.validation.accuracy",
    "metrics.validation.precision_macro",
    "metrics.validation.recall_macro",
    "metrics.validation.latency_per_row_ms",
    "run_id",
]].copy()

comparison = comparison.sort_values(
    "metrics.validation.f1_macro", ascending=False
).reset_index(drop=True)

display(comparison)
print(f"\n{len(comparison)} runs comparables encontrados.")

## 8. Aplica el gate sin mirar test

Filtra por F1 macro mínimo; ordena por F1 macro DESC, accuracy DESC y latencia ASC. Conserva
`BEST_RUN_ID`, `BEST_MODEL_URI` y un `selection_report` que diga explícitamente que test no
participó en la selección.

In [0]:
# TODO 8: aplica el gate sobre validation.f1_macro >= MIN_VALIDATION_F1_MACRO.
eligible = comparison[
    comparison["metrics.validation.f1_macro"] >= MIN_VALIDATION_F1_MACRO
].copy()

assert len(eligible) > 0, (
    f"Ningún candidato supera el gate F1 macro >= {MIN_VALIDATION_F1_MACRO}. "
    "No rebajes el umbral a posteriori."
)

# Ordenar: F1 macro DESC, accuracy DESC, latencia ASC
eligible = eligible.sort_values(
    by=["metrics.validation.f1_macro", "metrics.validation.accuracy", "metrics.validation.latency_per_row_ms"],
    ascending=[False, False, True],
).reset_index(drop=True)

BEST_RUN_ID = eligible.iloc[0]["run_id"]
BEST_MODEL_URI = f"runs:/{BEST_RUN_ID}/model"
BEST_CANDIDATE_NAME = eligible.iloc[0]["tags.candidate_name"]

selection_report = {
    "gate": f"validation.f1_macro >= {MIN_VALIDATION_F1_MACRO}",
    "tiebreakers": ["f1_macro DESC", "accuracy DESC", "latency ASC"],
    "best_run_id": BEST_RUN_ID,
    "best_candidate": BEST_CANDIDATE_NAME,
    "best_f1_macro": float(eligible.iloc[0]["metrics.validation.f1_macro"]),
    "best_accuracy": float(eligible.iloc[0]["metrics.validation.accuracy"]),
    "best_latency_ms": float(eligible.iloc[0]["metrics.validation.latency_per_row_ms"]),
    "num_eligible": int(len(eligible)),
    "num_total": int(len(comparison)),
    "test_participated_in_selection": False,
    "note": "Test NO participó en la selección del ganador.",
}

print(json.dumps(selection_report, indent=2))
print(f"\n🏆 Ganador: {BEST_CANDIDATE_NAME} (run {BEST_RUN_ID[:12]}...)")

## 9. Carga el modelo desde MLflow, evalúa test una sola vez y completa el run ganador

In [0]:
# TODO 9: load_model(BEST_MODEL_URI), calcula test.*, reabre BEST_RUN_ID,
# registra el input de testing, métricas, tags de selección y selection/decision.json.
best_model = mlflow.sklearn.load_model(BEST_MODEL_URI)

test_metrics = evaluate_classifier(best_model, X_test, y_test)
test_predictions = test_metrics.pop("predictions")

with mlflow.start_run(run_id=BEST_RUN_ID):
    # Log testing input
    test_df = X_test.copy()
    test_df["quality_class"] = y_test.values
    test_dataset = mlflow.data.from_pandas(
        test_df, source=DATASET_PATH, targets="quality_class", name="wine_test",
    )
    mlflow.log_input(test_dataset, context="testing")

    # Log test metrics
    mlflow.log_metrics({f"test.{k}": v for k, v in test_metrics.items()})

    # Tags de selección
    mlflow.set_tags({
        "selection.status": "winner",
        "selection.gate": f"f1_macro >= {MIN_VALIDATION_F1_MACRO}",
        "selection.test_participated": "false",
    })

    # Selection/decision JSON
    decision = {
        "action": "selected_as_winner",
        "run_id": BEST_RUN_ID,
        "candidate": BEST_CANDIDATE_NAME,
        "validation_f1_macro": float(eligible.iloc[0]["metrics.validation.f1_macro"]),
        "test_f1_macro": float(test_metrics["f1_macro"]),
        "test_accuracy": float(test_metrics["accuracy"]),
        "test_opened_after_selection": True,
        "test_used_for_selection": False,
    }
    with tempfile.NamedTemporaryFile(mode="w", suffix="_decision.json", delete=False) as f:
        json.dump(decision, f, indent=2)
        f.flush()
        mlflow.log_artifact(f.name, artifact_path="selection")

    # Classification report de test
    test_report = classification_report(
        y_test, test_predictions, output_dict=True, zero_division=0,
    )
    with tempfile.NamedTemporaryFile(mode="w", suffix="_test_report.json", delete=False) as f:
        json.dump(test_report, f, indent=2)
        f.flush()
        mlflow.log_artifact(f.name, artifact_path="metrics")

    # Confusion matrix de test
    labels_sorted = sorted(np.unique(np.concatenate([y_test.values, test_predictions])))
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test.values, test_predictions, labels=labels_sorted, ax=ax,
        display_labels=[class_mapping[l] for l in labels_sorted],
    )
    ax.set_title("Test Confusion Matrix - Winner")
    plt.tight_layout()
    mlflow.log_figure(fig, "artifacts/confusion_matrix_test.png")
    plt.close(fig)

print("📊 MÉTRICAS DE TEST (ganador):")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"\n🏆 Ganador: {BEST_CANDIDATE_NAME} | run: {BEST_RUN_ID[:12]}...")

## 10. Registra el modelo ganador en Unity Catalog

Usa el catálogo y esquema activos, un nombre de tres niveles y tu alias saneado. Añade
descripción y tags. Registra referencia+ganador, asigna `Challenger`, haz smoke test,
promociona, comprueba rollback y deja el ganador como `Champion`. Si recibes `PERMISSION_DENIED`, consulta la
guía: no pegues credenciales ni cambies a recursos de pago.

In [0]:
# TODO 10: set_registry_uri('databricks-uc'), register_model, descripciones,
# tags, Challenger, promoción, rollback, re-promoción y variable champion.
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

MODEL_NAME = f"workspace.default.wine_quality_{STUDENT_ALIAS}"

# Registrar el modelo ganador en UC
model_version = mlflow.register_model(
    model_uri=BEST_MODEL_URI,
    name=MODEL_NAME,
    await_registration_for=300,
)
version_number = model_version.version
print(f"✅ Registrado: {MODEL_NAME} v{version_number}")

# Descripción del modelo registrado
client.set_registered_model(
    name=MODEL_NAME,
    description=(
        f"Wine Quality multiclase (3..8). Semana 1 - Ciclo de vida con MLflow. "
        f"Alias: {STUDENT_ALIAS}. Batch: {BATCH_ID}. "
        f"Pipeline: SimpleImputer(median) + estimador."
    ),
)
client.set_model_version_tag(
    name=MODEL_NAME, version=version_number, key="student_alias", value=STUDENT_ALIAS
)
client.set_model_version_tag(
    name=MODEL_NAME, version=version_number, key="batch_id", value=BATCH_ID
)
client.set_model_version_tag(
    name=MODEL_NAME, version=version_number, key="model_type", value="winner"
)

# Asignar Challenger
client.set_model_version_alias(
    name=MODEL_NAME, version=version_number, alias="Challenger"
)
print(f"✅ {MODEL_NAME} v{version_number} asignado como Challenger")

# Smoke test: cargar por alias Challenger
challenger_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@Challenger")
_sample = X_test.head(3)
_smoke_pred = challenger_model.predict(_sample)
print(f"✅ Smoke test Challenger: predicciones = {_smoke_pred}")

# Promocionar a Champion
client.set_model_version_alias(
    name=MODEL_NAME, version=version_number, alias="Champion"
)
client.delete_model_version_alias(
    name=MODEL_NAME, version=version_number, alias="Challenger"
)
print(f"✅ {MODEL_NAME} v{version_number} promocionado a Champion")

# Probar rollback: quitar Champion, poner Challenger, luego restaurar
client.delete_model_version_alias(
    name=MODEL_NAME, version=version_number, alias="Champion"
)
client.set_model_version_alias(
    name=MODEL_NAME, version=version_number, alias="Challenger"
)
print(f"  ↩️ Rollback: Champion → Challenger (v{version_number})")

# Re-promoción
client.set_model_version_alias(
    name=MODEL_NAME, version=version_number, alias="Champion"
)
client.delete_model_version_alias(
    name=MODEL_NAME, version=version_number, alias="Challenger"
)
print(f"  ✅ Re-promoción: Challenger → Champion (v{version_number})")

champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@Champion")
print(f"✅ Champion cargado y verificado: {MODEL_NAME} v{version_number}")
REG_MODEL_NAME = MODEL_NAME
REG_MODEL_VERSION = version_number

## 11. Carga `models:/<nombre>@Champion` y realiza un smoke test con tres filas

In [0]:
# TODO 11: prueba que el consumidor puede cargar por alias y predecir tres filas.
loaded_by_alias = mlflow.sklearn.load_model(f"models:/{REG_MODEL_NAME}@Champion")

# Seleccionar 3 filas representativas (una por cada sextil del dataset)
_sample_rows = X_test.iloc[[0, len(X_test)//2, -1]].copy()
_preds = loaded_by_alias.predict(_sample_rows)
_probas = loaded_by_alias.predict_proba(_sample_rows)

print("🔍 Smoke test del Registry (models:/...@Champion):")
print(f"  Modelo cargado: {REG_MODEL_NAME} v{REG_MODEL_VERSION}")
for i in range(len(_sample_rows)):
    pred_class = int(_preds[i])
    pred_quality = class_mapping[pred_class]
    print(f"  Fila {i}: quality_class={pred_class} (quality={pred_quality})  proba_max={_probas[i].max():.3f}")

assert len(_preds) == 3, "El smoke test debe predecir 3 filas."
print("✅ Smoke test Registry superado.")

## 12. API local con el artefacto Joblib ganador

Resuelve el run de `Champion`, descarga `deployment/model.joblib` y levanta un
`ThreadingHTTPServer` en `127.0.0.1` con puerto automático. Implementa `GET /health` y
`POST /predict`; valida `instances`, columnas ausentes y columnas extra.

In [0]:
# TODO 12a: resuelve champion.run_id, descarga runs:/.../deployment/model.joblib,
# cárgalo con joblib.load y comprueba que predice una fila con EXPECTED_FEATURES.
client = MlflowClient()

# Obtener el run_id del Champion
mv = client.get_model_version_by_alias(REG_MODEL_NAME, "Champion")
champion_run_id = mv.run_id
print(f"Champion run_id: {champion_run_id}")

# Descargar el artefacto joblib
joblib_uri = f"runs:/{champion_run_id}/deployment/model.joblib"
local_path = mlflow.artifacts.download_artifacts(artifact_uri=joblib_uri)
print(f"Descargado: {local_path}")

# Cargar y verificar
joblib_model = joblib.load(local_path)
print(f"Modelo cargado: {type(joblib_model)}")

# Verificar que predice una fila con EXPECTED_FEATURES
_test_row = X_test.head(1)
_pred = joblib_model.predict(_test_row)
print(f"Predicción de prueba: quality_class={int(_pred[0])} (quality={class_mapping[int(_pred[0])]})")
assert hasattr(joblib_model, "predict"), "El modelo cargado debe tener método predict"
assert list(_test_row.columns) == EXPECTED_FEATURES, "Las columnas no coinciden con el contrato"
print("✅ Artefacto Joblib cargado y verificado.")

### 12b. Handler y contrato

Define `PredictionHandler` con `_send_json`, `GET /health` y `POST /predict`. Valida objeto raíz,
lista no vacía de objetos, columnas exactas y errores 400. Instrumenta contadores y latencias.

In [0]:
API_STATS = {"requests_total": 0, "requests_success": 0, "requests_rejected": 0, "requests_error": 0, "latencies_ms": []}
API_STATS_LOCK = threading.Lock()


class PredictionHandler(BaseHTTPRequestHandler):
    def _send_json(self, status_code: int, payload: dict):
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status_code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/health":
            self._send_json(200, {"status": "healthy"})
        else:
            self._send_json(404, {"error": "Not found"})

    def do_POST(self):
        t0 = time.perf_counter()
        with API_STATS_LOCK:
            API_STATS["requests_total"] += 1

        try:
            content_length = int(self.headers.get("Content-Length", 0))
            raw = self.rfile.read(content_length)
            body = json.loads(raw) if raw else None

            # Validar que el body es un dict
            if not isinstance(body, dict):
                with API_STATS_LOCK:
                    API_STATS["requests_rejected"] += 1
                self._send_json(400, {"error": "Request body must be a JSON object"})
                return

            # Validar instances
            instances = body.get("instances")
            if not isinstance(instances, list) or len(instances) == 0:
                with API_STATS_LOCK:
                    API_STATS["requests_rejected"] += 1
                self._send_json(400, {"error": "'instances' must be a non-empty list of objects"})
                return

            # Construir DataFrame
            try:
                df = pd.DataFrame(instances)
            except Exception:
                with API_STATS_LOCK:
                    API_STATS["requests_rejected"] += 1
                self._send_json(400, {"error": "Could not build DataFrame from instances"})
                return

            # Validar columnas exactas
            missing = [c for c in EXPECTED_FEATURES if c not in df.columns]
            extra = [c for c in df.columns if c not in EXPECTED_FEATURES]
            if missing or extra:
                with API_STATS_LOCK:
                    API_STATS["requests_rejected"] += 1
                self._send_json(400, {
                    "error": "Column mismatch",
                    "missing": missing,
                    "extra": extra,
                    "expected": EXPECTED_FEATURES,
                })
                return

            # Reordenar columnas
            df = df[EXPECTED_FEATURES]

            # Predecir
            preds = joblib_model.predict(df)
            probas = joblib_model.predict_proba(df)
            results = []
            for i in range(len(preds)):
                results.append({
                    "quality_class": int(preds[i]),
                    "quality": class_mapping[int(preds[i])],
                    "probabilities": {
                        str(class_mapping[j]): float(probas[i][j])
                        for j in range(len(probas[i]))
                    },
                })

            elapsed_ms = (time.perf_counter() - t0) * 1000
            with API_STATS_LOCK:
                API_STATS["requests_success"] += 1
                API_STATS["latencies_ms"].append(elapsed_ms)

            self._send_json(200, {"predictions": results})

        except Exception as e:
            with API_STATS_LOCK:
                API_STATS["requests_error"] += 1
            self._send_json(500, {"error": str(e)})

    def log_message(self, format, *args):
        pass  # Silenciar logs de servidor

### 12c. Arranque y limpieza idempotente

Arranca en `127.0.0.1` y puerto 0. Define `stop_api()` antes de probar para que un `finally` pueda
cerrar servidor, socket e hilo. No uses `0.0.0.0`.

In [0]:
# TODO 12c: crea api_server, API_PORT, api_thread, API_STOPPED y stop_api().
api_server = ThreadingHTTPServer(("127.0.0.1", 0), PredictionHandler)
API_PORT = api_server.server_address[1]
api_thread = threading.Thread(target=api_server.serve_forever, daemon=True)
api_thread.start()
API_STOPPED = False

def stop_api():
    global API_STOPPED
    if not API_STOPPED:
        api_server.shutdown()
        api_server.server_close()
        api_thread.join(timeout=5)
        API_STOPPED = True

print(f"✅ API corriendo en http://127.0.0.1:{API_PORT}")
print(f"   GET  /health  -> 200 healthy")
print(f"   POST /predict -> predicciones multiclase")

## 13. Pruebas de contrato y de error

In [0]:
# TODO 13: crea call_json con try/finally para apagar siempre la API y verifica:
# GET /health -> 200; POST válido -> 200; sólo alcohol -> 400; JSON no-object -> 400.
def call_json(method: str, path: str, body=None) -> tuple[int, dict]:
    url = f"http://127.0.0.1:{API_PORT}{path}"
    data = json.dumps(body).encode("utf-8") if body is not None else None
    req = Request(url, data=data, method=method)
    req.add_header("Content-Type", "application/json")
    try:
        with urlopen(req) as resp:
            return resp.status, json.loads(resp.read())
    except HTTPError as e:
        return e.code, json.loads(e.read())


try:
    # 1. GET /health -> 200
    status, body = call_json("GET", "/health")
    assert status == 200, f"GET /health esperado 200, recibido {status}"
    assert body["status"] == "healthy"
    print(f"✅ GET /health -> {status}: {body}")

    # 2. POST válido -> 200
    valid_instances = X_test.head(3).to_dict(orient="records")
    status, body = call_json("POST", "/predict", {"instances": valid_instances})
    assert status == 200, f"POST /predict válido esperado 200, recibido {status}"
    assert len(body["predictions"]) == 3
    print(f"✅ POST /predict (válido) -> {status}: {len(body['predictions'])} predicciones")
    print(f"   Ejemplo: quality={body['predictions'][0]['quality']}")

    # 3. POST con sólo alcohol -> 400 (columnas faltantes)
    partial_instances = [{"alcohol": 9.4}]
    status, body = call_json("POST", "/predict", {"instances": partial_instances})
    assert status == 400, f"POST /predict parcial esperado 400, recibido {status}"
    assert "missing" in body or "error" in body
    print(f"✅ POST /predict (sólo alcohol) -> {status}: columnas faltantes rechazadas")

    # 4. POST con JSON no-object (lista) -> 400
    status, body = call_json("POST", "/predict", [1, 2, 3])
    assert status == 400, f"POST /predict no-object esperado 400, recibido {status}"
    print(f"✅ POST /predict (JSON no-object) -> {status}: rechazado")

    print("\n✅ Todas las pruebas de API superadas.")

finally:
    stop_api()
    print("✅ API detenida.")

## 14. Registra un run separado para la evidencia del despliegue

In [0]:
# TODO 14: registra versión/alias/run origen, requests totales, éxitos, rechazadas,
# latencias y deployment/evidence.json. Conserva DEPLOYMENT_RUN_ID.
# No registres el payload de entrada: la evidencia debe contener sólo agregados.
with mlflow.start_run(
    experiment_id=experiment_id,
    run_name=f"{STUDENT_ALIAS}_{BATCH_ID}_deployment",
) as deploy_run:
    DEPLOYMENT_RUN_ID = deploy_run.info.run_id

    mlflow.set_tags({
        "student_alias": STUDENT_ALIAS,
        "batch_id": BATCH_ID,
        "workflow": "s01-lifecycle",
        "run_type": "deployment",
        "model_name": REG_MODEL_NAME,
        "model_version": str(REG_MODEL_VERSION),
        "champion_run_id": champion_run_id,
    })

    mlflow.log_param("mlflow_version", mlflow.__version__)
    mlflow.log_param("sklearn_version", sklearn.__version__)
    mlflow.log_param("xgboost_version", xgboost.__version__)

    # Agregados de la API (sólo agregados, no payloads)
    with API_STATS_LOCK:
        stats = dict(API_STATS)
    latencies = stats.pop("latencies_ms", [])

    mlflow.log_metric("requests_total", stats.get("requests_total", 0))
    mlflow.log_metric("requests_success", stats.get("requests_success", 0))
    mlflow.log_metric("requests_rejected", stats.get("requests_rejected", 0))
    mlflow.log_metric("requests_error", stats.get("requests_error", 0))

    if latencies:
        mlflow.log_metric("latency_mean_ms", float(np.mean(latencies)))
        mlflow.log_metric("latency_p50_ms", float(np.percentile(latencies, 50)))
        mlflow.log_metric("latency_p95_ms", float(np.percentile(latencies, 95)))
        mlflow.log_metric("latency_max_ms", float(np.max(latencies)))

    evidence = {
        "model_name": REG_MODEL_NAME,
        "model_version": REG_MODEL_VERSION,
        "champion_run_id": champion_run_id,
        "deployment_run_id": DEPLOYMENT_RUN_ID,
        "api_stats": stats,
        "latency_summary_ms": {
            "mean": float(np.mean(latencies)) if latencies else 0,
            "p50": float(np.percentile(latencies, 50)) if latencies else 0,
            "p95": float(np.percentile(latencies, 95)) if latencies else 0,
            "max": float(np.max(latencies)) if latencies else 0,
        },
        "tests_passed": [
            "GET /health -> 200",
            "POST /predict (válido) -> 200",
            "POST /predict (sólo alcohol) -> 400",
            "POST /predict (JSON no-object) -> 400",
        ],
    }
    with tempfile.NamedTemporaryFile(mode="w", suffix="_evidence.json", delete=False) as f:
        json.dump(evidence, f, indent=2)
        f.flush()
        mlflow.log_artifact(f.name, artifact_path="deployment")

    print(f"✅ Run de despliegue registrado: {DEPLOYMENT_RUN_ID}")
    print(json.dumps(evidence, indent=2))

## 15. Apaga la API y entrega el resumen del ciclo

In [0]:
# TODO 15: shutdown, server_close, join y assert de que el hilo terminó.
# Imprime experiment_id, batch, ganador, modelo/versión/alias y deployment run.
if not API_STOPPED:
    stop_api()

assert API_STOPPED, "La API debe estar detenida al finalizar."
assert not api_thread.is_alive(), "El hilo del servidor aún está vivo."

summary = {
    "experiment_id": experiment_id,
    "batch_id": BATCH_ID,
    "student_alias": STUDENT_ALIAS,
    "num_candidates": len(CANDIDATES),
    "num_runs": len(candidate_run_ids),
    "best_run_id": BEST_RUN_ID,
    "best_candidate": BEST_CANDIDATE_NAME,
    "model_name": REG_MODEL_NAME,
    "model_version": REG_MODEL_VERSION,
    "model_alias": "Champion",
    "deployment_run_id": DEPLOYMENT_RUN_ID,
    "test_metrics": {k: round(v, 4) for k, v in test_metrics.items()},
}

print("=" * 70)
print("📋 RESUMEN DEL CICLO DE VIDA")
print("=" * 70)
print(json.dumps(summary, indent=2))
print("=" * 70)
print("✅ Notebook completado: datos → experimentos → gate → ganador → Registry → API → observabilidad")